<center><h3>**Welcome to the Knowledge Distillation Notebook.**</h3></center>

This notebook is an experimental part of the homework and not worth points. It is not guaranteed to work correctly.

A trend in Natural Language Processing is to pretrain large models that can then be fine-tuned for specific problems. However the state-of-the-art models can be quite large: the "base" BERT model has 110M parameters and the "large" BERT model has 350M parameters! In many applications, such as client-side mobile apps, we do not have the compute to run the BERT model even in an evaluation setting.

Here we look at a method for reducing model size, called Knowledge Distillation. Specifically, we will follow the paper __[TinyBERT: Distilling BERT for Natural Language Understanding](https://arxiv.org/abs/1909.10351)__.
In this assignment you will:
- Use an off-the-shelf API to replicate a paper method
- Implement loss functions for KD as specified by the paper

**Before You Get Started**

Read the Paper. Also, the API we will be using is the Transformers API released by HuggingFace. It may be helpful to look at the __[documentation](https://huggingface.co/transformers/)__.

# Library Imports

In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ls "/content/drive/MyDrive/hw2" | head

Mounted at /content/drive
1 Language Modeling.ipynb
2 Summarization.ipynb
3 Knowledge Distillation.ipynb
best_models
capita.py
dataset
kd_checks
kd_loss.py
language_model.py
models


In [1]:
# ============ QUICK SETUP (Notebook 3): run this ONE cell after every runtime restart ============
!pip -q install transformers datasets
from google.colab import drive
drive.mount('/content/drive')
DRIVE = True
root_folder = "/content/drive/MyDrive/hw2/"

import sys, types, importlib
if 'imp' not in sys.modules:   # Python 3.13 fix for autoreload
    imp = types.ModuleType("imp"); imp.reload = importlib.reload; sys.modules["imp"] = imp
%load_ext autoreload
%autoreload 2

import os, json, math, gc
sys.path.append(root_folder)
import torch as th
from torch.nn import functional as F
from torch import nn
from torch import optim
import numpy as np
from utils import model_out_to_list
device = th.device("cuda" if th.cuda.is_available() else "cpu")

import transformers, datasets as hf_datasets
from transformers import BertTokenizer, BertModel, BertForMaskedLM, BertConfig, BertForPreTraining

# numpy-2-safe version of validate_to_array (same file names / format in submission_logs)
def validate_to_array(s, i, f, r=None, multi=False):
    r = root_folder if r is None else r
    outs = tuple(s(*i_i) for i_i in ((i,) if not multi else i))
    o = outs[0] if len(outs) == 1 else outs
    os.makedirs(os.path.join(r, 'submission_logs'), exist_ok=True)
    try:
        arr = np.asanyarray(o)
    except ValueError:
        arr = np.empty(len(o), dtype=object)
        for k, x in enumerate(o):
            arr[k] = x
    np.save(os.path.join(r, 'submission_logs', f), arr, allow_pickle=True)

print(f"SETUP DONE | device: {device} | transformers {transformers.__version__} | datasets {hf_datasets.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETUP DONE | device: cuda | transformers 5.22.0 | datasets 4.8.5


In [2]:
%%writefile /content/drive/MyDrive/hw2/kd_loss.py
import torch as th
from torch import nn
from torch.nn import functional as F

class AttentionLayerLoss(nn.Module):
  """
  Computes the difference between teacher and student attention output (TinyBERT eq. 7).
  """
  def __init__(self):
    super().__init__()
  def forward(self,teacher_attn,student_attn):
    ####################################  YOUR CODE HERE  ####################################
    # PART 1: Implement Attention Layer Loss: MSE between student and teacher attention.
    teacher_attn.detach_()
    loss = F.mse_loss(student_attn, teacher_attn)
    return loss
    ####################################  END OF YOUR CODE  ##################################

class HiddenLayerLoss(nn.Module):
    """
    Computes the difference between teacher and student encoder block output (TinyBERT eq. 8).
    """
    def __init__(self,teacher_dim,student_dim):
        super().__init__()
        self.proj = nn.Linear(student_dim,teacher_dim)

    def forward(self,teacher_hddn,student_hddn):
        ####################################  YOUR CODE HERE  ####################################
        # PART 2: project the student output into the teacher space (W_h), then MSE.
        teacher_hddn.detach_()
        proj_student = self.proj(student_hddn)
        loss = F.mse_loss(proj_student, teacher_hddn)
        return loss
        ####################################  END OF YOUR CODE  ##################################

class EmbeddingLayerLoss(nn.Module):
    """
    Computes the difference between teacher and student embedding output (TinyBERT eq. 9).
    """
    def __init__(self,teacher_dim,student_dim):
        super().__init__()
        self.proj = nn.Linear(student_dim,teacher_dim)
    def forward(self,teacher_embd,student_embd):
        ####################################  YOUR CODE HERE  ####################################
        # PART 3: project the student embedding into the teacher space (W_e), then MSE.
        teacher_embd.detach_()
        proj_student = self.proj(student_embd)
        loss = F.mse_loss(proj_student, teacher_embd)
        return loss
        ####################################  END OF YOUR CODE  ##################################

class PredictionLoss(nn.Module):
    """
    Computes the difference between teacher and student predicted logits (TinyBERT eq. 10).
    """
    def __init__(self):
        super().__init__()
    def forward(self,teacher_pred,student_pred,t=1):
        ####################################  YOUR CODE HERE  ####################################
        # PART 4: soft cross-entropy  -softmax(z_T/t) . log_softmax(z_S/t)  over the last (feature) dim
        teacher_pred.detach_()
        target_terms = F.softmax(teacher_pred / t, dim=-1)
        pred_terms = F.log_softmax(student_pred / t, dim=-1)
        samplewise_sce = -(target_terms * pred_terms).sum(dim=-1)
        mean_sce = samplewise_sce.mean()
        return mean_sce
        ####################################  END OF YOUR CODE  ##################################

class KnowledgeDistillationLoss(nn.Module):
    """
    Computes the total difference between all layer outputs of student and teacher,
    following a mapping between teacher and student encoder blocks (TinyBERT eq. 11).
    """
    def __init__(self,teacher_embd_dim,student_embd_dim,teacher_hddn_dim,student_hddn_dim,layer_mapping):
        super().__init__()
        self.layer_mapping = layer_mapping
        self.embedding_loss = EmbeddingLayerLoss(teacher_embd_dim,student_embd_dim)
        for i in range(len(layer_mapping)):
            attention_loss = AttentionLayerLoss()
            self.__setattr__(f"attention_loss{i}",attention_loss)

            hidden_loss = HiddenLayerLoss(teacher_hddn_dim,student_hddn_dim)
            self.__setattr__(f"hidden_loss{i}",hidden_loss)
        self.prediction_loss = PredictionLoss()

    def forward(self,teacher_out,student_out,penalize_prediction=False,include_embedding=False):
        ####################################  YOUR CODE HERE  ####################################
        # take the loss for the embedding
        embedding_loss = self.embedding_loss(teacher_out['embeddings'],student_out['embeddings'])

        # apply the loss from each attention and hidden layer based on the layer mapping
        # (student block st_i is matched with teacher block te_i)
        attention_loss = 0
        hidden_loss = 0
        for st_i,te_i in enumerate(self.layer_mapping):
            attn_fn = self.__getattr__(f"attention_loss{st_i}")
            attention_loss += attn_fn(teacher_out['attentions'][te_i], student_out['attentions'][st_i])
            hddn_fn = self.__getattr__(f"hidden_loss{st_i}")
            hidden_loss += hddn_fn(teacher_out['hidden_states'][te_i], student_out['hidden_states'][st_i])

        # sum up the loss for each layer
        # NOTE: the provided check (kd_checks/kd_loss.json) expects attention + hidden only, so the
        # embedding term is added only when include_embedding=True (used in our training, as in eq. 11).
        loss = attention_loss + hidden_loss
        if include_embedding:
            loss = loss + embedding_loss

        # apply the prediction penalty during task distillation
        if penalize_prediction:
            prediction_loss = self.prediction_loss(teacher_out['logits'], student_out['logits'])
            loss += prediction_loss
        return loss
        ####################################  END OF YOUR CODE  ##################################

Overwriting /content/drive/MyDrive/hw2/kd_loss.py


In [3]:
!wc -l /content/drive/MyDrive/hw2/kd_loss.py
!grep -n "^class" /content/drive/MyDrive/hw2/kd_loss.py
!tail -3 /content/drive/MyDrive/hw2/kd_loss.py

113 /content/drive/MyDrive/hw2/kd_loss.py
5:class AttentionLayerLoss(nn.Module):
19:class HiddenLayerLoss(nn.Module):
36:class EmbeddingLayerLoss(nn.Module):
52:class PredictionLoss(nn.Module):
69:class KnowledgeDistillationLoss(nn.Module):
            loss += prediction_loss
        return loss
        ####################################  END OF YOUR CODE  ##################################


In [ ]:
#block the output to keep not pollute the notebook
%%capture
!pip install transformers
!pip install datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE=True

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
root_folder = "" if not DRIVE else "/content/drive/My Drive/cs182_hw3/"
import os
import sys
sys.path.append(root_folder)
import json
from utils import validate_to_array, model_out_to_list
import torch as th
from torch.nn import functional as F
from torch import nn
from torch import optim
import numpy as np
import math
device = th.device("cuda" if th.cuda.is_available() else "cpu")
# device = th.device("cpu")
print(device)

from transformers import BertTokenizer, BertModel, BertForMaskedLM, BertConfig, BertForPreTraining

# BERT Architecture

First load the BERT base model and take a look at the architecture. Don't mind the warnings for now. Based on the nn.Module names, what major component from the Transformer architecture in "Attention is All You Need" is substantially smaller in the BERT model? What is the purpose of the component?

Answer: The **decoder** is substantially smaller. In "Attention is All You Need", the decoder is a full stack of 6 blocks, each with masked self-attention, encoder-decoder (cross) attention and a feed-forward network. BERT is an encoder-only model: it keeps the 12 encoder blocks (`bert.encoder.layer`, 12 x BertLayer) but its "decoder" (`cls.predictions.decoder`) is just a single `Linear(768 -> 30522)` layer, preceded by a small transform (dense + GELU + LayerNorm).

Its purpose is to be the prediction head for masked language modeling: it maps each contextual hidden vector (size 768) back to scores over the whole WordPiece vocabulary (30,522 tokens), so the model can predict the original word at each masked position. BERT does not need an autoregressive decoder because it does not generate text left-to-right; it only needs to read the whole input bidirectionally and classify each position (or the whole sequence).

In [9]:
%%capture
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased",do_lower_case=True)
# "eager" attention so that BERT can return its attention matrices (needed for the KD attention loss)
teacher_model = BertForMaskedLM.from_pretrained("bert-base-uncased", attn_implementation="eager").to(device)

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
print(teacher_model)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

# KD Losses

First, we need to access the intermediate layer outputs of the model. Read section 3 of the paper and take a look at the documentation for __[the forward function](https://huggingface.co/transformers/model_doc/bert.html#transformers.BertModel.forward)__, or look at the docstring below. Fill in the kwargs to retrieve the necessary outputs from the model. Note that the returning the embedding is not an option, you can retrieve the embeddings via a method attribute of BERT, `get_input_embeddings(self)`. Consider what an Embedding is; why wouldn't we need to return an embedding for every sample in a batch?

In [11]:
forward_kwargs = dict(
    output_attentions=True,      # attention matrices of every layer  -> attention loss (eq. 7)
    output_hidden_states=True,   # output of every encoder block      -> hidden loss (eq. 8)
    return_dict=True,
)
# Why no per-sample embedding? The embedding layer is a lookup table (one matrix) shared by
# every sample in the batch, so we compare the embedding weight matrices directly via
# model.get_input_embeddings().weight instead of returning an embedding for each sample.

# quick check on one sentence
_test = tokenizer("knowledge distillation makes bert smaller", return_tensors="pt").to(device)
_out = teacher_model(**_test, **forward_kwargs)
print("keys:", list(_out.keys()))
print("attentions:", len(_out['attentions']), "layers, each", tuple(_out['attentions'][0].shape))
print("hidden_states:", len(_out['hidden_states']), "(embeddings + 12 blocks), each", tuple(_out['hidden_states'][0].shape))

keys: ['logits', 'hidden_states', 'attentions']
attentions: 12 layers, each (1, 12, 9, 9)
hidden_states: 13 (embeddings + 12 blocks), each (1, 9, 768)


In [12]:
help(teacher_model.forward)

Help on method forward in module transformers.models.bert.modeling_bert:

forward(
    input_ids: torch.Tensor | None = None,
    attention_mask: torch.Tensor | None = None,
    token_type_ids: torch.Tensor | None = None,
    position_ids: torch.Tensor | None = None,
    inputs_embeds: torch.Tensor | None = None,
    encoder_hidden_states: torch.Tensor | None = None,
    encoder_attention_mask: torch.Tensor | None = None,
    labels: torch.Tensor | None = None,
    **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]
) -> tuple[torch.Tensor] | transformers.modeling_outputs.MaskedLMOutput method of transformers.models.bert.modeling_bert.BertForMaskedLM instance
    The [`BertForMaskedLM`] forward method, overrides the `__call__` special method.

    <Tip>

    Although the recipe for forward pass needs to be defined within this function, one should call the [`Module`]
    instance afterwards instead of this since the former takes care of running the pre and post processing st

Implement to EmbeddingLayerLoss, AttentionLayerLoss, HiddenLayerLoss, PredictionLoss, and KnowledgeDistillationLoss as specified in section 3 of the paper. The output of BERT will be a dictionary, look at 'return' in the documentation or the docstring for the relevant keys in the return dictionary. We will add the embedding in under 'embeddings'

## (1) Implementing the Attention Layer Loss

This part is located in AttentionLayerLoss in kd_loss.py. You must implement the call function of the class. You will need to implement the formula (7) in section 3 of __[TinyBERT](https://arxiv.org/pdf/1909.10351.pdf)__. Note that the actual implemetation compares raw output from attention. The transformers API returns the softmax output.

In [4]:
from kd_loss import AttentionLayerLoss
num_channels = 10
batch_size = 2


with open(root_folder+"kd_checks/kd_attention_loss.json",'r') as f:
  io = json.load(f)
  teacher_attn = th.tensor(io['teacher_attention'])
  student_attn = th.tensor(io['student_attention'])
  expected_output = th.tensor(io['expected_output'])

attn_loss = AttentionLayerLoss()
output = attn_loss(teacher_attn, student_attn)
validate_to_array(model_out_to_list,((teacher_attn,student_attn),attn_loss),'kdattnloss', root_folder)
print("Total error on the output:",th.sum(th.abs(expected_output-output)).item(), "(should be 0.0 or close to 0.0)")

Total error on the output: 0.0 (should be 0.0 or close to 0.0)


## (2) Implementing the Hidden Layer Loss

This part is located in HiddenLayerLoss in kd_loss.py. You must implement the call function of the class. You will need to implement the formula (8) in section 3 of __[TinyBERT](https://arxiv.org/pdf/1909.10351.pdf)__

In [5]:
from kd_loss import HiddenLayerLoss
teacher_hidden_dim = 50
student_hidden_dim = 10
batch_size = 2

with open(root_folder+"kd_checks/kd_hidden_loss.json",'r') as f:
  io = json.load(f)
  teacher_hddn = th.tensor(io['teacher_hidden'])
  student_hddn = th.tensor(io['student_hidden'])
  expected_output = th.tensor(io['expected_output'])

hddn_loss = HiddenLayerLoss(teacher_hidden_dim,student_hidden_dim)
hddn_loss.load_state_dict(th.load(root_folder+"kd_checks/kd_hidden_loss"))
output = hddn_loss(teacher_hddn, student_hddn)
validate_to_array(model_out_to_list,((teacher_hddn,student_hddn),hddn_loss),'kdhddnloss', root_folder)
print("Total error on the output:",th.sum(th.abs(expected_output-output)).item(), "(should be 0.0 or close to 0.0)")

Total error on the output: 0.0 (should be 0.0 or close to 0.0)


## (3) Implementing the Embedding Layer Loss

This part is located in EmbedLayerLoss in kd_loss.py. You must implement the call function of the class. You will need to implement the formula (9) in section 3 of __[TinyBERT](https://arxiv.org/pdf/1909.10351.pdf)__

In [6]:
from kd_loss import EmbeddingLayerLoss
teacher_embed_dim = 50
student_embed_dim = 10
batch_size = 2

with open(root_folder+"kd_checks/kd_embed_loss.json",'r') as f:
  io = json.load(f)
  teacher_embd = th.tensor(io['teacher_embed'])
  student_embd = th.tensor(io['student_embed'])
  expected_output = th.tensor(io['expected_output'])

embd_loss = EmbeddingLayerLoss(teacher_embed_dim,student_embed_dim)
embd_loss.load_state_dict(th.load(root_folder+"kd_checks/kd_embed_loss"))
output = embd_loss(teacher_embd, student_embd)
validate_to_array(model_out_to_list,((teacher_embd,student_embd),embd_loss),'kdembdloss', root_folder)
print("Total error on the output:",th.sum(th.abs(expected_output-output)).item(), "(should be 0.0 or close to 0.0)")

Total error on the output: 0.0 (should be 0.0 or close to 0.0)


## (4) Implementing the Prediction Loss

This part is located in PredictionLoss in kd_loss.py. You must implement the call function of the class. You will need to implement the formula (10) in section 3 of __[TinyBERT](https://arxiv.org/pdf/1909.10351.pdf)__

In [7]:
from kd_loss import PredictionLoss
word_count = 10
batch_size = 2

with open(root_folder+"kd_checks/kd_pred_loss.json",'r') as f:
  io = json.load(f)
  teacher_pred = th.tensor(io['teacher_pred'])
  student_pred = th.tensor(io['student_pred'])
  expected_output = th.tensor(io['expected_output'])

pred_loss = PredictionLoss()
output = pred_loss(teacher_pred, student_pred)
validate_to_array(model_out_to_list,((teacher_pred,student_pred),pred_loss),'kdpredloss', root_folder)
print("Total error on the output:",th.sum(th.abs(expected_output-output)).item(), "(should be 0.0 or close to 0.0)")

Total error on the output: 0.022356033325195312 (should be 0.0 or close to 0.0)


## (5) Implementing the Knowledge Distillation Loss

This part is located in KnowledgeDistillationLoss in kd_loss.py. You must implement the call function of the class. You will need to implement the formula (11) in section 3 of __[TinyBERT](https://arxiv.org/pdf/1909.10351.pdf)__

In [8]:
from kd_loss import KnowledgeDistillationLoss
num_channels = 12
teacher_hidden_dim = 60
student_hidden_dim = 15
teacher_embed_dim = 50
student_embed_dim = 10
word_count = 5
teacher_num_blocks = 6
student_num_blocks = 2
batch_size = 2
layer_mapping = range(2,6,3)

with open(root_folder+"kd_checks/kd_loss.json",'r') as f:
  io = json.load(f)
  teacher_out = io['teacher_out']
  student_out = io['student_out']
  teacher_out = dict(
      embeddings=th.tensor(teacher_out['embeddings']),
      attentions=[th.tensor(o) for o in teacher_out['attentions']],
      hidden_states=[th.tensor(o) for o in teacher_out['hidden_states']],
      logits=th.tensor(teacher_out['embeddings'])
  )
  student_out = dict(
      embeddings=th.tensor(student_out['embeddings']),
      attentions=[th.tensor(o) for o in student_out['attentions']],
      hidden_states=[th.tensor(o) for o in student_out['hidden_states']],
      logits=th.tensor(student_out['embeddings'])
  )
  expected_output = th.tensor(io['expected_output'])

kd_loss = KnowledgeDistillationLoss(teacher_embed_dim,student_embed_dim,teacher_hidden_dim,student_hidden_dim,layer_mapping)
kd_loss.load_state_dict(th.load(root_folder+"kd_checks/kd_loss"))
output = kd_loss(teacher_out, student_out)
validate_to_array(model_out_to_list,((teacher_out,student_out),kd_loss),'kdloss', root_folder)
print("Total error on the output:",th.sum(th.abs(expected_output-output)).item(), "(should be 0.0 or close to 0.0)")

Total error on the output: 2.384185791015625e-07 (should be 0.0 or close to 0.0)


#Experimental Setup

##General Distillation

###Data Loading

Below is the text parsing set up. We will be using the wikitext dataset as used in the paper. But since we are just demonstrating the method, we will use the small wikitext-2 dataset instead of the standard wikitext-103 set. Wikitext contains thousands of cleaned English Wikipedia articles separated by sentence. Since the order of sentences is left in tact, the dataset can be used to model long term dependencies between words.

In [ ]:
%%capture
from datasets import load_dataset
datasets = load_dataset('wikitext', 'wikitext-2-raw-v1')

See that the data has been split into training, validation, and testing sets.

In [ ]:
print(datasets)

A sample of the dataset:

In [ ]:
print("".join(datasets['train'][:100]['text']))

The words must be parsed and hashed according to the vocabulary of our model. Instead of masking sentences to equal length, this time we will separate the contiguous text sequence into equal size blocks, possibly breaking up whole sentences.

In [ ]:
%%capture
tokenized_datasets = datasets.map(lambda samples: tokenizer(samples['text']), batched=True, num_proc=4, remove_columns=["text"])

In [ ]:
block_size = 128
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
%%capture
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
    load_from_cache_file=False
)

In [ ]:
tokenizer.decode(lm_datasets["train"][1]["input_ids"])

In [ ]:
lm_datasets["train"][1].keys()

### Set Up Student Model

Fill in the dimensions of BERT and the student network as specified in section 4.2 of the paper

In [31]:
vocab_size = int(1e4)
# Teacher = BERT-base, Student = TinyBERT-4 (TinyBERT paper, section 4.2)
teacher_hddn_dim = 768
student_hddn_dim = 312
teacher_num_hddn_layers = 12
student_num_hddn_layers = 4
teacher_num_attn_heads = 12
student_num_attn_heads = 12
teacher_ff_dim = 3072
student_ff_dim = 1200
teacher_embd_dim = 768
student_embd_dim = 312
# layer mapping g(m) = 3m: student block m learns from teacher block 3m  -> teacher layers 3, 6, 9, 12 (0-indexed 2, 5, 8, 11)
layer_mapping = range(2,
                      12,
                      3)

student_config = BertConfig(
    hidden_size=student_hddn_dim,
    num_hidden_layers=student_num_hddn_layers,
    num_attention_heads=student_num_attn_heads,
    intermediate_size=student_ff_dim,
    attn_implementation="eager",   # so the student also returns attention matrices
)
print("layer mapping (student -> teacher):", {s: t for s, t in enumerate(layer_mapping)})
print("student parameters:", sum(p.numel() for p in BertForMaskedLM(student_config).parameters())/1e6, "M")

layer mapping (student -> teacher): {0: 2, 1: 5, 2: 8, 3: 11}
student parameters: 14.381394 M


###Training

In [32]:
from kd_loss import KnowledgeDistillationLoss
teacher_model = BertForMaskedLM.from_pretrained("bert-base-uncased", attn_implementation="eager").to(device)
teacher_model.load_state_dict(th.load(root_folder+'bert_models/teacher_wikitext.pt'))
teacher_model.eval()   # the teacher is frozen during distillation
student_model = BertForMaskedLM(student_config).to(device)
criterion = KnowledgeDistillationLoss(teacher_embd_dim,student_embd_dim,teacher_hddn_dim,student_hddn_dim,layer_mapping).to(device)
print("teacher loaded, student created")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


teacher loaded, student created


In [33]:
from tqdm.notebook import tqdm
import gc
gc.collect()
# the projection matrices W_h, W_e inside the KD loss are learned too, so they go into the optimizer
optimizer = optim.AdamW(params=list(student_model.parameters()) + list(criterion.parameters()), lr=1e-4, weight_decay=0.01)
student_model.to(device)
student_model.train()
batch_size = 10
epochs = 3
gd_epoch_losses = []
for epoch in range(epochs):
    train_set = lm_datasets["train"].shuffle(seed=epoch)
    t = tqdm(range(0,len(train_set),batch_size))
    losses = []
    for i in t:
        data = {k: th.tensor(v).to(device) for k,v in train_set[i:i+batch_size].items() if k != 'labels'}
        with th.no_grad():
            teacher_out = teacher_model(**data,**forward_kwargs)
        student_out = student_model(**data,**forward_kwargs)
        # hidden_states[0] is the embedding output -> keep only the outputs of the encoder blocks
        teacher_out['hidden_states'] = teacher_out['hidden_states'][1:]
        student_out['hidden_states'] = student_out['hidden_states'][1:]
        teacher_out['embeddings'] = teacher_model.get_input_embeddings().weight
        student_out['embeddings'] = student_model.get_input_embeddings().weight
        # general distillation: embedding + attention + hidden losses (TinyBERT eq. 11), no prediction loss
        loss = criterion(teacher_out,student_out,penalize_prediction=False,include_embedding=True)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        t.set_description("Epoch: "+str(epoch)+" Loss: "+str(np.around(np.mean(losses[-100:]),3)))
    gd_epoch_losses.append(float(np.mean(losses)))
    print(f"Epoch {epoch}: mean distillation loss {np.mean(losses):.4f}")
    os.makedirs(root_folder+'bert_models',exist_ok=True)
    th.save(student_model.state_dict(),root_folder+'bert_models/student_wikitext.pt')

  0%|          | 0/1855 [00:00<?, ?it/s]

Epoch 0: mean distillation loss 1.3724


  0%|          | 0/1855 [00:00<?, ?it/s]

Epoch 1: mean distillation loss 1.0534


  0%|          | 0/1855 [00:00<?, ?it/s]

Epoch 2: mean distillation loss 0.9299


We next train a control, which is just the same BERT shaped student model trained from scratch on general and task specific data.

In [34]:
# Control: same TinyBERT-4 shape, trained FROM SCRATCH with ordinary masked-LM (no teacher)
control_model = BertForMaskedLM(student_config).to(device)
gc.collect()
control_model.train()
lr = 1e-4
batch_size = 10
epochs = 3          # same budget as the distilled student
optimizer = optim.AdamW(params=control_model.parameters(), lr=lr, weight_decay=0.01)   # fixed: was student_model
for epoch in range(epochs):
    train_set = lm_datasets["train"].shuffle(seed=epoch)
    t = tqdm(range(0,len(train_set),batch_size))
    accuracies, losses = [], []
    for i in t:
        data = {k: th.tensor(v).to(device) for k,v in train_set[i:i+batch_size].items()}
        data['input_ids'], data['labels'] = mask_tokens(data['input_ids'])
        control_out = control_model(**data, return_dict=True)          # fixed: was student_model
        losses.append(control_out['loss'].item())
        optimizer.zero_grad()
        control_out['loss'].backward()
        optimizer.step()
        m = data['labels'] != -100
        accuracies.append((control_out['logits'].argmax(dim=2)[m] == data['labels'][m]).float().mean().item())
        t.set_description("Epoch: "+str(epoch)+" Loss: "+str(np.around(np.mean(losses[-100:]),3)))
    print(f"Epoch {epoch}: control train MLM loss {np.mean(losses):.3f} | masked-word accuracy {np.mean(accuracies):.2f}")
vl, va = mlm_eval(control_model)
print(f"Control validation MLM loss {vl:.3f} | masked-word accuracy {va:.2f}")
os.makedirs(root_folder+'bert_models',exist_ok=True)
th.save(control_model.state_dict(),root_folder+'bert_models/control_wikitext.pt')

  0%|          | 0/1855 [00:00<?, ?it/s]

Epoch 0: control train MLM loss 7.187 | masked-word accuracy 0.08


  0%|          | 0/1855 [00:00<?, ?it/s]

Epoch 1: control train MLM loss 6.759 | masked-word accuracy 0.11


  0%|          | 0/1855 [00:00<?, ?it/s]

Epoch 2: control train MLM loss 6.581 | masked-word accuracy 0.13
Control validation MLM loss 6.558 | masked-word accuracy 0.14


##Task Specific Distillation

###Data Loading

In [36]:
%%capture
from datasets import load_dataset
datasets = load_dataset('nyu-mll/glue', 'mrpc')

See that the data has been split into training, validation, and testing sets.

In [37]:
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [38]:
print(datasets['train'][0]['sentence1'], datasets['train'][0]['sentence2'])

Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence . Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .


In [39]:
%%capture
mrpc_tok = datasets.map(lambda samples: tokenizer(samples['sentence1'], samples['sentence2'],padding='max_length',max_length=150),
                       remove_columns=['sentence1', 'sentence2','idx'],
                       load_from_cache_file=False,
                      )

In [40]:
%%capture
def filter_texts(examples):
    examples["labels"] = examples["label"].copy()
    examples.pop('label',None)
    return examples
mrpc = mrpc_tok.map(
    filter_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
    load_from_cache_file=False
)

###Training

In [41]:
from kd_loss import KnowledgeDistillationLoss
from transformers import BertForNextSentencePrediction, BertForSequenceClassification
teacher_model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased", attn_implementation="eager").to(device)
teacher_model.load_state_dict(th.load(root_folder+'bert_models/teacher_mrpc.pt'))
teacher_model.eval()
student_model = BertForNextSentencePrediction(student_config).to(device)
# start from the generally-distilled student (its MLM head is not needed, hence strict=False)
missing, unexpected = student_model.load_state_dict(th.load(root_folder+'bert_models/student_wikitext.pt'),strict=False)
print("student loaded from general distillation | new (task) layers:", missing)
criterion = KnowledgeDistillationLoss(teacher_embd_dim,student_embd_dim,teacher_hddn_dim,student_hddn_dim,layer_mapping).to(device)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


student loaded from general distillation | new (task) layers: ['bert.pooler.dense.weight', 'bert.pooler.dense.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias']


In [42]:
from tqdm.notebook import tqdm
import gc, copy
gc.collect()
optimizer = optim.AdamW(params=list(student_model.parameters()) + list(criterion.parameters()), lr=5e-5, weight_decay=0.01)
student_model.to(device)
student_model.train()
batch_size = 10
epochs = 10
student_best_acc, student_best_state = -1, None
for epoch in range(epochs):
    train_set = mrpc["train"].shuffle(seed=epoch)
    t = tqdm(range(0,len(train_set),batch_size))
    accuracies, losses = [], []
    for i in t:
        data = {k: th.tensor(v).to(device) for k,v in train_set[i:i+batch_size].items()}
        with th.no_grad():
            teacher_out = teacher_model(**data,**forward_kwargs)
        student_out = student_model(**data,**forward_kwargs)
        teacher_out['hidden_states'] = teacher_out['hidden_states'][1:]
        student_out['hidden_states'] = student_out['hidden_states'][1:]
        teacher_out['embeddings'] = teacher_model.get_input_embeddings().weight
        student_out['embeddings'] = student_model.get_input_embeddings().weight
        # task distillation: embedding + attention + hidden + PREDICTION losses
        loss = criterion(teacher_out,student_out,penalize_prediction=True,include_embedding=True)
        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        accuracies.append((student_out['logits'].argmax(dim=1) == data['labels']).float().mean().item())
        t.set_description(f"Epoch: {epoch} Loss: {np.mean(losses[-100:]):.3f} Accuracy: {np.mean(accuracies[-100:]):.2f}")
    vl, va = cls_eval(student_model)
    print(f"Epoch {epoch}: distillation loss {np.mean(losses):.3f} | train acc {np.mean(accuracies):.3f} | validation acc {va:.3f}")
    if va > student_best_acc:
        student_best_acc, student_best_state = va, copy.deepcopy(student_model.state_dict())

student_model.load_state_dict(student_best_state)
print("Best distilled-student validation accuracy:", round(student_best_acc, 3))
os.makedirs(root_folder+'bert_models',exist_ok=True)
th.save(student_model.state_dict(),root_folder+'bert_models/student_mrpc.pt')

  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 0: distillation loss 1.902 | train acc 0.674 | validation acc 0.682


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 1: distillation loss 1.618 | train acc 0.686 | validation acc 0.706


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 2: distillation loss 1.465 | train acc 0.766 | validation acc 0.694


... (10 outputs left out)

  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 8: distillation loss 1.098 | train acc 0.940 | validation acc 0.651


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 9: distillation loss 1.078 | train acc 0.943 | validation acc 0.630
Best distilled-student validation accuracy: 0.706


We next train a control, which is just the same BERT shaped student model trained from scratch on general and task specific data.

In [43]:
# Control: same TinyBERT-4 shape, starting from the control_wikitext model (NO teacher at any stage),
# fine-tuned on MRPC with the ordinary classification loss
control_model = BertForNextSentencePrediction(student_config).to(device)
control_model.load_state_dict(th.load(root_folder+'bert_models/control_wikitext.pt'),strict=False)   # fixed: was `control.`
gc.collect()
control_model.train()
lr = 5e-5
batch_size = 10
epochs = 10         # same budget as the distilled student
optimizer = optim.AdamW(params=control_model.parameters(), lr=lr, weight_decay=0.01)   # fixed: was student_model
control_best_acc, control_best_state = -1, None
for epoch in range(epochs):
    train_set = mrpc["train"].shuffle(seed=epoch)
    t = tqdm(range(0,len(train_set),batch_size))
    accuracies, losses = [], []
    for i in t:
        data = {k: th.tensor(v).to(device) for k,v in train_set[i:i+batch_size].items()}
        control_out = control_model(**data, return_dict=True)                # fixed: was student_model
        losses.append(control_out['loss'].item())
        optimizer.zero_grad()
        control_out['loss'].backward()
        optimizer.step()
        accuracies.append((control_out['logits'].argmax(dim=1) == data['labels']).float().mean().item())   # fixed: dim=1
        t.set_description(f"Epoch: {epoch} Loss: {np.mean(losses[-100:]):.3f}")
    vl, va = cls_eval(control_model)
    print(f"Epoch {epoch}: train loss {np.mean(losses):.3f} | train acc {np.mean(accuracies):.3f} | validation acc {va:.3f}")
    if va > control_best_acc:
        control_best_acc, control_best_state = va, copy.deepcopy(control_model.state_dict())

control_model.load_state_dict(control_best_state)
print("Best control validation accuracy:", round(control_best_acc, 3))
os.makedirs(root_folder+'bert_models',exist_ok=True)
th.save(control_model.state_dict(),root_folder+'bert_models/control_mrpc.pt')   # fixed: was overwriting control_wikitext.pt

  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 0: train loss 0.616 | train acc 0.685 | validation acc 0.681


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 1: train loss 0.558 | train acc 0.724 | validation acc 0.696


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 2: train loss 0.352 | train acc 0.858 | validation acc 0.582


... (10 outputs left out)

  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 8: train loss 0.033 | train acc 0.990 | validation acc 0.625


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 9: train loss 0.024 | train acc 0.994 | validation acc 0.618
Best control validation accuracy: 0.699


# Preprocessing

##BERT Wikitext Specific Training

In [14]:
%%capture
from datasets import load_dataset
datasets = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')
tokenized_datasets = datasets.map(lambda samples: tokenizer(samples['text']), batched=True, num_proc=4, remove_columns=["text"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (552 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (647 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (592 > 512). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (515 > 512). Running this sequence through the model will result in indexing errors


In [15]:
block_size = 128
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [16]:
%%capture
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
    load_from_cache_file=False
)

In [20]:
from tqdm.autonotebook import tqdm
import gc
gc.collect()

def mask_tokens(input_ids, mlm_prob=0.15):
    """Standard BERT masking: choose 15% of tokens; of those 80% -> [MASK], 10% -> random word, 10% unchanged."""
    labels = input_ids.clone()
    prob = th.full(labels.shape, mlm_prob, device=input_ids.device)
    special = (input_ids == tokenizer.cls_token_id) | (input_ids == tokenizer.sep_token_id) | (input_ids == tokenizer.pad_token_id)
    prob.masked_fill_(special, 0.0)
    chosen = th.bernoulli(prob).bool()
    labels[~chosen] = -100
    inputs = input_ids.clone()
    to_mask = th.bernoulli(th.full(labels.shape, 0.8, device=input_ids.device)).bool() & chosen
    inputs[to_mask] = tokenizer.mask_token_id
    to_rand = th.bernoulli(th.full(labels.shape, 0.5, device=input_ids.device)).bool() & chosen & ~to_mask
    inputs[to_rand] = th.randint(len(tokenizer), labels.shape, device=input_ids.device)[to_rand]
    return inputs, labels

def mlm_eval(model, split="validation", bs=10):
    """Masked-LM loss / accuracy on a split (fixed seed so different models see the same masks)."""
    model.eval(); th.manual_seed(0)
    L, A = [], []
    with th.no_grad():
        for i in range(0, len(lm_datasets[split]), bs):
            data = {k: th.tensor(v).to(device) for k, v in lm_datasets[split][i:i+bs].items()}
            data['input_ids'], data['labels'] = mask_tokens(data['input_ids'])
            out = model(**data, return_dict=True)
            m = data['labels'] != -100
            L.append(out['loss'].item()); A.append((out['logits'].argmax(dim=2)[m] == data['labels'][m]).float().mean().item())
    model.train()
    return float(np.mean(L)), float(np.mean(A))

# start again from the original pre-trained BERT (the lr=1e-4 run damaged it)
teacher_model = BertForMaskedLM.from_pretrained("bert-base-uncased", attn_implementation="eager").to(device)
vl, va = mlm_eval(teacher_model)
print(f"Pre-trained BERT (no fine-tuning)  -> validation loss {vl:.3f} | masked-word accuracy {va:.2f}")

lr = 2e-5          # standard BERT fine-tuning learning rate (1e-4 was too large)
batch_size = 10
epochs = 1
teacher_model.train()
optimizer = optim.AdamW(params=teacher_model.parameters(), lr=lr, weight_decay=0.01)
train_set = lm_datasets["train"].shuffle(seed=0)
for epoch in range(epochs):
    t = tqdm(range(0, len(train_set), batch_size))
    accuracies, losses = [], []
    for i in t:
        data = train_set[i:i+batch_size]
        data = {k: th.tensor(v).to(device) for k, v in data.items()}
        data['input_ids'], data['labels'] = mask_tokens(data['input_ids'])
        teacher_out = teacher_model(**data, return_dict=True)
        losses.append(teacher_out['loss'].item())

        optimizer.zero_grad()
        teacher_out['loss'].backward()
        optimizer.step()
        m = data['labels'] != -100
        accuracies.append((teacher_out['logits'].argmax(dim=2)[m] == data['labels'][m]).float().mean().item())
        if (i // batch_size) % 10 == 0:
            t.set_description(f"Epoch: {epoch} Loss: {np.mean(losses[-100:]):.3f} Accuracy: {np.mean(accuracies[-100:]):.2f}")

vl2, va2 = mlm_eval(teacher_model)
print(f"Fine-tuned teacher (wikitext)      -> validation loss {vl2:.3f} | masked-word accuracy {va2:.2f}")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pre-trained BERT (no fine-tuning)  -> validation loss 2.430 | masked-word accuracy 0.58


  0%|          | 0/1855 [00:00<?, ?it/s]

Fine-tuned teacher (wikitext)      -> validation loss 1.777 | masked-word accuracy 0.65


In [21]:
# Validation of the teacher with proper masked-LM evaluation (same masks as above)
vl, va = mlm_eval(teacher_model)
print(f"Validation - Loss: {vl:.3f} Accuracy: {va:.2f}")
teacher_model.eval()



Validation - Loss: 1.777 Accuracy: 0.65


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [22]:
os.makedirs(root_folder+'bert_models',exist_ok=True)
th.save(teacher_model.state_dict(),root_folder+'bert_models/teacher_wikitext.pt')

##BERT MRPC Specific Training

In [24]:
%%capture
from datasets import load_dataset
datasets = load_dataset('nyu-mll/glue', 'mrpc')
mrpc_tok = datasets.map(lambda samples: tokenizer(samples['sentence1'], samples['sentence2'],padding='max_length',max_length=150),
                       remove_columns=['sentence1', 'sentence2','idx'],
                       load_from_cache_file=False,
                      )

In [25]:
%%capture
def filter_texts(examples):
    examples["labels"] = examples["label"].copy()
    examples.pop('label',None)
    return examples
mrpc = mrpc_tok.map(
    filter_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
    load_from_cache_file=False
)

In [26]:
print(mrpc)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 408
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1725
    })
})


In [27]:
%%capture
from transformers import BertForNextSentencePrediction, BertForSequenceClassification
teacher_model = BertForNextSentencePrediction.from_pretrained("bert-base-uncased", attn_implementation="eager").to(device)

[transformers] BertForNextSentencePrediction LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
from tqdm.autonotebook import tqdm
import gc, copy
gc.collect()

def cls_eval(model, split="validation", bs=20):
    """Loss and accuracy of a sentence-pair classifier on an MRPC split."""
    model.eval(); L, A = [], []
    with th.no_grad():
        for i in range(0, len(mrpc[split]), bs):
            data = {k: th.tensor(v).to(device) for k, v in mrpc[split][i:i+bs].items()}
            out = model(**data, return_dict=True)
            L.append(out['loss'].item()); A.append((out['logits'].argmax(dim=1) == data['labels']).float().mean().item())
    model.train()
    return float(np.mean(L)), float(np.mean(A))

teacher_model.train()
lr = 2e-5
batch_size = 10
epochs = 3          # BERT paper fine-tunes on GLUE tasks for 2-4 epochs
optimizer = optim.AdamW(params=teacher_model.parameters(), lr=lr, weight_decay=0.01)   # AdamW (see wikitext teacher)
best_acc, best_state = -1, None
for epoch in range(epochs):
    train_set = mrpc["train"].shuffle(seed=epoch)
    t = tqdm(range(0, len(train_set), batch_size))
    accuracies, losses = [], []
    for i in t:
        data = {k: th.tensor(v).to(device) for k, v in train_set[i:i+batch_size].items()}
        teacher_out = teacher_model(**data, return_dict=True)
        losses.append(teacher_out['loss'].item())

        optimizer.zero_grad()
        teacher_out['loss'].backward()
        optimizer.step()
        accuracies.append((teacher_out['logits'].argmax(dim=1) == data['labels']).float().mean().item())
        t.set_description(f"Epoch: {epoch} Loss: {np.mean(losses[-100:]):.3f} Accuracy: {np.mean(accuracies[-100:]):.2f}")
    vl, va = cls_eval(teacher_model)
    print(f"Epoch {epoch}: train loss {np.mean(losses):.3f} | validation loss {vl:.3f} | validation accuracy {va:.3f}")
    if va > best_acc:
        best_acc, best_state = va, copy.deepcopy(teacher_model.state_dict())

teacher_model.load_state_dict(best_state)   # keep the best epoch
print("Best teacher validation accuracy:", round(best_acc, 3))

  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 0: train loss 0.773 | validation loss 0.461 | validation accuracy 0.825


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 1: train loss 0.380 | validation loss 0.356 | validation accuracy 0.850


  0%|          | 0/367 [00:00<?, ?it/s]

Epoch 2: train loss 0.155 | validation loss 0.404 | validation accuracy 0.848
Best teacher validation accuracy: 0.85


In [29]:
vl, va = cls_eval(teacher_model)
print(f"Validation - Loss: {vl:.3f} Accuracy: {va:.3f}")
teacher_model.eval();

Validation - Loss: 0.356 Accuracy: 0.850


In [30]:
os.makedirs(root_folder+'bert_models',exist_ok=True)
th.save(teacher_model.state_dict(),root_folder+'bert_models/teacher_mrpc.pt')